In [1]:
# ==============================================================================
# PHẦN 1: IMPORT THƯ VIỆN VÀ ĐỌC DỮ LIỆU
# ==============================================================================
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import backend as K

# Ẩn các cảnh báo không cần thiết
warnings.filterwarnings('ignore')

# Hiển thị tối đa 100 cột khi xem dữ liệu
pd.set_option('display.max_columns', 100)

# Đường dẫn dữ liệu trên Kaggle
train_file = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
test_file = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'

# Nếu không chạy trên Kaggle thì đọc file trong thư mục hiện tại
train = pd.read_csv(train_file if os.path.exists(train_file) else 'train.csv')
test = pd.read_csv(test_file if os.path.exists(test_file) else 'test.csv')

# Kiểm tra số dòng và số cột của hai tập dữ liệu
print(f"Kích thước ban đầu - Train: {train.shape}, Test: {test.shape}")

# Lưu cột Id để dùng khi tạo file submission
test_id = test['Id']

Kích thước ban đầu - Train: (1460, 81), Test: (1459, 80)


In [2]:
# ==============================================================================
# PHẦN 2: XỬ LÝ DỮ LIỆU
# ==============================================================================

# 1. Xử lý dữ liệu bị thiếu trong tập train
cat_col_train = [
    'FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual',
    'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'GarageQual', 'GarageCond'
]
ncat_col_train = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']

# Cột phân loại được điền bằng giá trị xuất hiện nhiều nhất
for col in cat_col_train:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].mode()[0])

# Cột số được điền bằng giá trị trung bình
for col in ncat_col_train:
    if col in train.columns:
        train[col] = train[col].fillna(train[col].mean())

# 2. Làm tương tự với tập test
cat_col_test = [
    'FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual',
    'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'GarageQual', 'GarageCond', 'MSZoning', 'Utilities', 'Exterior1st',
    'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType'
]
ncat_col_test = [
    'LotFrontage', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
    'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'GarageCars', 'GarageArea'
]

for col in cat_col_test:
    if col in test.columns:
        test[col] = test[col].fillna(test[col].mode()[0])

for col in ncat_col_test:
    if col in test.columns:
        test[col] = test[col].fillna(test[col].mean())

# 3. Bỏ cột Id và các cột bị thiếu quá nhiều dữ liệu
to_drop = ['Id', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']
train.drop(columns=to_drop, inplace=True)
test.drop(columns=to_drop, inplace=True)

# 4. Gộp train và test để các cột sau khi mã hóa giống nhau
final_df = pd.concat([train, test], axis=0)

all_cat_col = [
    'MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
    'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
    'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd',
    'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
    'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating',
    'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PavedDrive', 'SaleType', 'SaleCondition'
]

# Chuyển các cột phân loại thành các cột số bằng One-Hot Encoding
def cat_onehot_encoding(multicol, df):
    df_final = df.copy()

    for field in multicol:
        if field in df_final.columns:
            df1 = pd.get_dummies(df_final[field], drop_first=True)
            df_final.drop([field], axis=1, inplace=True)
            df_final = pd.concat([df_final, df1], axis=1)

    return df_final


final_df = cat_onehot_encoding(all_cat_col, final_df)

# Bỏ các cột bị trùng tên sau khi mã hóa
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# 5. Tách dữ liệu trở lại thành train và test
df_train = final_df.iloc[:1460, :]
df_test = final_df.iloc[1460:, :].copy()

# Tập test không cần cột giá bán
df_test.drop(['SalePrice'], axis=1, inplace=True)

# Tách biến đầu vào và biến cần dự đoán
X_train = df_train.drop(['SalePrice'], axis=1)
y_train = df_train['SalePrice']

# Chuyển các đặc trưng về cùng kiểu dữ liệu
X_train = X_train.astype(float)
X_test = df_test.astype(float)

# Kiểm tra lại kích thước dữ liệu sau khi xử lý
print(
    f"Kích thước sau tiền xử lý - "
    f"X_train: {X_train.shape}, X_test: {X_test.shape}"
)

Kích thước sau tiền xử lý - X_train: (1460, 175), X_test: (1459, 175)
